# YgoMaster-PackBuilder — Installer Tests

Test offline di `install_ygomaster_build()` usando esclusivamente directory temporanee.

Copre installazione completa, patch di `Shop.json`, idempotenza, CRLF/BOM, file estranei, build invalida, target mancanti e valori `PerPackRarities` invalidi/duplicati.


In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

from ygomaster_packbuilder.domain import (
    Card, OpeningPolicy, PackDefinition, PackType, ProjectDefinition,
    Rarity, RarityDistribution, SourcePackRef, SourceProvider,
    TenPackGuarantee, UltraRarePity,
)
from ygomaster_packbuilder.exporters.ygomaster import write_ygomaster_build
from ygomaster_packbuilder.installer import (
    YgoMasterInstallError, YgoMasterInstallResult, install_ygomaster_build,
)

def make_policy():
    return OpeningPolicy(
        key="configuration-a",
        normal_slot_distribution=RarityDistribution(55.0,35.0,7.5,2.5),
        ten_pack_guarantee=TenPackGuarantee(
            True, RarityDistribution(0.0,0.0,80.0,20.0)
        ),
        ultra_rare_pity=UltraRarePity(
            True, RarityDistribution(0.0,0.0,0.0,100.0)
        ),
    )

def make_card(name, ydk_id, md_id, rarity):
    return Card(
        name=name,
        ydk_id=ydk_id,
        source_rarity=rarity.value,
        master_duel_id=md_id,
        rarity=rarity,
    )

def make_pack(key, pack_type, booster_size, cards):
    expected = {
        PackType.PERSONAL_MAIN: 100,
        PackType.PERSONAL_MINI: 20,
        PackType.SHARED_STAPLES: 200,
    }[pack_type]
    return PackDefinition(
        key=key,
        name=key,
        description=f"{key} description",
        type=pack_type,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key,
        ),
        expected_card_count=expected,
        booster_size=booster_size,
        opening_policy_key="configuration-a",
        cards=cards,
    )

project = ProjectDefinition(
    name="Installer Test",
    packs=[
        make_pack(
            "main-pack", PackType.PERSONAL_MAIN, 7,
            [make_card("Normal",1,1001,Rarity.N), make_card("Ultra",2,1002,Rarity.UR)]
        ),
        make_pack(
            "mini-pack", PackType.PERSONAL_MINI, 3,
            [make_card("Rare",3,1003,Rarity.R), make_card("Super",4,1004,Rarity.SR)]
        ),
    ],
    opening_policies=[make_policy()],
)

pack_ids = {"main-pack":9_000_000, "mini-pack":9_000_001}

BASE_SHOP = '''{
    "DefaultSecretDuration": 86400,
    // this comment must survive
    "NoDuplicatesPerPack": true,
    "PackShop": {}
}
'''

def make_build(root):
    build_dir = Path(root) / "build"
    artifacts = write_ygomaster_build(project, pack_ids, build_dir)
    return build_dir, artifacts

def make_target(root, shop_text):
    ygomaster_dir = Path(root) / "YgoMaster"
    data_dir = ygomaster_dir / "Data"
    data_dir.mkdir(parents=True)
    (data_dir / "Shop.json").write_bytes(shop_text.encode("utf-8"))
    return ygomaster_dir

print("Fixtures ready")

Fixtures ready


## 1. Installazione completa

In [2]:
with TemporaryDirectory() as tmp:
    build_dir, build_artifacts = make_build(tmp)
    ygomaster_dir = make_target(tmp, BASE_SHOP)

    result = install_ygomaster_build(
        project, pack_ids, build_dir, ygomaster_dir
    )

    assert isinstance(result, YgoMasterInstallResult)

    expected_shop = ygomaster_dir / "Data" / "Shop" / "ygomaster-packbuilder.json"
    expected_odds = ygomaster_dir / "Data" / "ShopPackOdds" / "ygomaster-packbuilder.json"
    main_shop = ygomaster_dir / "Data" / "Shop.json"

    assert expected_shop.read_bytes() == build_artifacts.shop.read_bytes()
    assert expected_odds.read_bytes() == build_artifacts.shop_pack_odds.read_bytes()
    assert set(result.changed_files) == {expected_shop, expected_odds, main_shop}

print("Complete installation OK")

Complete installation OK


## 2. Chiave assente / false / true

In [3]:
# assente
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(tmp, BASE_SHOP)
    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    text = (ygomaster_dir / "Data" / "Shop.json").read_text(encoding="utf-8")
    assert '"PerPackRarities": true,' in text
    assert text.count('"PerPackRarities"') == 1
    assert "// this comment must survive" in text

# false
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(
        tmp,
        '{\n    "PerPackRarities": false,\n    "PackShop": {}\n}\n'
    )
    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    text = (ygomaster_dir / "Data" / "Shop.json").read_text(encoding="utf-8")
    assert '"PerPackRarities": true' in text
    assert '"PerPackRarities": false' not in text

# already true
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(
        tmp,
        '{\n    "PerPackRarities": true,\n    "PackShop": {}\n}\n'
    )
    shop_path = ygomaster_dir / "Data" / "Shop.json"
    before = shop_path.read_bytes()
    result = install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    assert shop_path.read_bytes() == before
    assert shop_path not in result.changed_files

print("PerPackRarities patch cases OK")

PerPackRarities patch cases OK


## 3. Secondo run idempotente

In [4]:
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(tmp, BASE_SHOP)

    first = install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    second = install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    assert len(first.changed_files) == 3
    assert second.changed_files == ()

print("Idempotence OK")

Idempotence OK


## 4. CRLF e UTF-8 BOM

In [5]:
# CRLF
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = Path(tmp) / "YgoMaster"
    data_dir = ygomaster_dir / "Data"
    data_dir.mkdir(parents=True)
    shop_path = data_dir / "Shop.json"
    shop_path.write_bytes(
        (
            "{\r\n"
            '    "DefaultSecretDuration": 86400,\r\n'
            "    // comment\r\n"
            '    "PackShop": {}\r\n'
            "}\r\n"
        ).encode("utf-8")
    )

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    raw = shop_path.read_bytes()

    assert b'{\r\n    "PerPackRarities": true,' in raw
    assert b"\r\n" in raw

# BOM
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = Path(tmp) / "YgoMaster"
    data_dir = ygomaster_dir / "Data"
    data_dir.mkdir(parents=True)
    shop_path = data_dir / "Shop.json"
    shop_path.write_bytes(BASE_SHOP.encode("utf-8-sig"))

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    raw = shop_path.read_bytes()

    assert raw.startswith(b"\xef\xbb\xbf")
    assert raw.count(b"\xef\xbb\xbf") == 1

print("CRLF + BOM preserved")

CRLF + BOM preserved


## 5. File estranei e `.tmp`

In [6]:
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(tmp, BASE_SHOP)

    other_shop = ygomaster_dir / "Data" / "Shop" / "other-mod.json"
    other_odds = ygomaster_dir / "Data" / "ShopPackOdds" / "other-mod.json"
    other_shop.parent.mkdir(parents=True, exist_ok=True)
    other_odds.parent.mkdir(parents=True, exist_ok=True)
    other_shop.write_text("keep shop", encoding="utf-8")
    other_odds.write_text("keep odds", encoding="utf-8")

    install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)

    assert other_shop.read_text(encoding="utf-8") == "keep shop"
    assert other_odds.read_text(encoding="utf-8") == "keep odds"

    temp_files = [
        p for p in ygomaster_dir.rglob("*")
        if p.is_file() and p.name.endswith(".ygomaster-packbuilder.tmp")
    ]
    assert temp_files == []

print("Unrelated files preserved; no tmp files")

Unrelated files preserved; no tmp files


## 6. Build invalida → target intatto

In [7]:
with TemporaryDirectory() as tmp:
    build_dir, artifacts = make_build(tmp)
    ygomaster_dir = make_target(tmp, BASE_SHOP)

    shop_path = ygomaster_dir / "Data" / "Shop.json"
    before = shop_path.read_bytes()

    artifacts.shop.write_text('{"PackShop": {}}', encoding="utf-8")

    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError as exc:
        print("Expected:", exc)
    else:
        raise AssertionError("Invalid build should be rejected")

    assert shop_path.read_bytes() == before
    assert not (ygomaster_dir / "Data" / "Shop" / "ygomaster-packbuilder.json").exists()
    assert not (ygomaster_dir / "Data" / "ShopPackOdds" / "ygomaster-packbuilder.json").exists()

print("Invalid build rejected before target writes")

Expected: Cannot install an invalid build:
- build-shop-odds-reference-mismatch: PackShop IDs and ShopPackOdds references do not match.
- build-shop-content-mismatch: Shop export does not match the expected export.
Invalid build rejected before target writes


## 7. Target mancanti

In [8]:
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)

    # root mancante
    try:
        install_ygomaster_build(project, pack_ids, build_dir, Path(tmp) / "missing")
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Missing root not rejected")

    # Data mancante
    ygomaster_dir = Path(tmp) / "YgoMaster"
    ygomaster_dir.mkdir()
    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Missing Data not rejected")

with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = Path(tmp) / "YgoMaster"
    (ygomaster_dir / "Data").mkdir(parents=True)

    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Missing Shop.json not rejected")

print("Missing target components rejected")

Missing target components rejected


## 8. `PerPackRarities` invalido / duplicato

In [9]:
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(
        tmp,
        '{\n    "PerPackRarities": 123,\n    "PackShop": {}\n}\n'
    )
    shop_path = ygomaster_dir / "Data" / "Shop.json"
    before = shop_path.read_bytes()

    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Invalid PerPackRarities not rejected")

    assert shop_path.read_bytes() == before
    assert not (ygomaster_dir / "Data" / "Shop" / "ygomaster-packbuilder.json").exists()

with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(
        tmp,
        '{\n'
        '    "PerPackRarities": false,\n'
        '    "PerPackRarities": true,\n'
        '    "PackShop": {}\n'
        '}\n'
    )

    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Duplicate PerPackRarities not rejected")

print("Invalid/duplicate settings rejected")

Invalid/duplicate settings rejected


## 9. `Shop.json` non-object

In [10]:
with TemporaryDirectory() as tmp:
    build_dir, _ = make_build(tmp)
    ygomaster_dir = make_target(tmp, "[]\n")

    try:
        install_ygomaster_build(project, pack_ids, build_dir, ygomaster_dir)
    except YgoMasterInstallError:
        pass
    else:
        raise AssertionError("Non-object Shop.json not rejected")

print("Non-object Shop.json rejected")

Non-object Shop.json rejected
